
# Experiment 8: Classification Model – Handwritten Digit Recognition

## Aim
To build, train, evaluate, and compare a Logistic Regression model
and a Dense Neural Network for recognizing handwritten digits
from 0 to 9 using the MNIST dataset.

## Objectives
- Load and explore the MNIST dataset.
- Normalize pixel values and flatten images.
- Train a Logistic Regression baseline model.
- Build and train a Dense Neural Network.
- Evaluate both models using accuracy, precision, recall, F1-score,
  and a confusion matrix.
- Compare model performance and interpret the results.

## Dataset Description
The MNIST dataset contains 70,000 grayscale images of handwritten
digits. Each image has dimensions of 28 × 28 pixels.

- Training samples: 60,000
- Testing samples: 10,000
- Input image size: 28 × 28 pixels
- Number of classes: 10 (digits 0–9)
- Pixel intensity range: 0–255

## Tools and Libraries
Python, NumPy, Matplotlib, Seaborn, Scikit-learn,
TensorFlow/Keras, and Jupyter Notebook.


In [ ]:
print("python working")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import mnist
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)



## 1. Load and Explore the Dataset

The MNIST dataset is loaded using TensorFlow/Keras.
It is divided into training and testing datasets.

The training dataset is used to train the models, while
the testing dataset is used to evaluate their performance.

MNIST is a multiclass classification problem because the model
must classify each image into one of ten digit classes.


In [ ]:

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training images shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing images shape:", X_test.shape)
print("Testing labels shape:", y_test.shape)
print("Pixel range:", X_train.min(), X_train.max())



## 2. Visualize Sample Images

Ten sample images are displayed with their corresponding labels.
This helps us understand the appearance of handwritten digits
and identify similarities between different classes.


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()



### Observation

The dataset contains handwritten digits with different shapes,
sizes, and stroke patterns. Digits with distinct shapes are
easier to recognize, while visually similar digits may be
misclassified.



## 3. Data Preprocessing

Pixel values are normalized by dividing them by 255, converting
the range from 0–255 to 0–1.

Each 28 × 28 image is then flattened into a one-dimensional
vector containing 784 features. Flattening changes the shape
of each image but does not change the number of samples.


In [ ]:

# Normalize pixel values
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten images into 784 features
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print("Flattened training shape:", X_train_flat.shape)
print("Flattened testing shape:", X_test_flat.shape)



## 4. Model A: Baseline Logistic Regression

Logistic Regression is used as the baseline classification model.
It learns the relationship between the 784 pixel features and
the ten possible digit classes.

The model is trained using the training dataset and evaluated
on the test dataset using classification metrics.


In [ ]:

model_lr = LogisticRegression(
    max_iter=100,
    solver="lbfgs"
)

model_lr.fit(X_train_flat, y_train)

y_pred_lr = model_lr.predict(X_test_flat)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, digits=4))



### Baseline Results

Accuracy measures the proportion of correctly classified images.
Precision measures the correctness of positive predictions for
a class, recall measures how many actual samples of that class
are correctly identified, and F1-score is their harmonic mean.

Macro averages calculate the unweighted mean across all ten classes.


In [ ]:

lr_accuracy = accuracy_score(y_test, y_pred_lr)

lr_precision = precision_score(
    y_test, y_pred_lr, average="macro", zero_division=0
)

lr_recall = recall_score(
    y_test, y_pred_lr, average="macro", zero_division=0
)

lr_f1 = f1_score(
    y_test, y_pred_lr, average="macro", zero_division=0
)

report_lr = classification_report(
    y_test, y_pred_lr,
    output_dict=True,
    zero_division=0
)

recalls_lr = {
    digit: report_lr[str(digit)]["recall"]
    for digit in range(10)
}

print("Accuracy:", round(lr_accuracy, 4))
print("Macro Precision:", round(lr_precision, 4))
print("Macro Recall:", round(lr_recall, 4))
print("Macro F1-score:", round(lr_f1, 4))

print("Most difficult class:",
      min(recalls_lr, key=recalls_lr.get))

print("Least difficult class:",
      max(recalls_lr, key=recalls_lr.get))



## 5. Confusion Matrix Analysis

A confusion matrix compares the actual labels with the predicted
labels. The diagonal represents correct predictions, while
off-diagonal entries represent misclassifications.

For digit 5, the problem can be treated as binary classification
using a one-vs-rest approach:

- True Positive (TP): Actual 5 predicted as 5.
- False Positive (FP): Other digits predicted as 5.
- False Negative (FN): Actual 5 predicted as another digit.
- True Negative (TN): Other digits correctly predicted as not 5.


In [ ]:

cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

# Digit with the highest number of correct predictions
print("Digit with highest correct predictions:",
      np.argmax(np.diag(cm_lr)))

# Most frequent directed confusion
cm_off = cm_lr.copy()
np.fill_diagonal(cm_off, 0)

i, j = np.unravel_index(np.argmax(cm_off), cm_off.shape)

print(f"Most frequent confusion: {i} predicted as {j}")
print("Number of errors:", cm_off[i, j])

# TP, FP, FN, TN for digit 5
tp = cm_lr[5, 5]
fn = cm_lr[5, :].sum() - tp
fp = cm_lr[:, 5].sum() - tp
tn = cm_lr.sum() - tp - fp - fn

print("\nMetrics for digit 5")
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)



### Interpretation

The diagonal values indicate correctly classified images.
The largest diagonal value identifies the digit with the most
correct predictions.

The largest off-diagonal value identifies the most frequent
directed confusion between two digits.

Accuracy alone is insufficient because it does not show the
individual performance of each class. Precision, recall,
F1-score, and the confusion matrix provide a more complete
evaluation of the model.



## 6. Model B: Dense Neural Network

A Dense Neural Network learns nonlinear relationships between
input pixels and digit classes.

The architecture consists of:
- Input layer: 784 features.
- Hidden layer 1: 128 neurons with ReLU activation.
- Dropout layer: 20% dropout to reduce overfitting.
- Hidden layer 2: 64 neurons with ReLU activation.
- Output layer: 10 neurons with Softmax activation.

The Adam optimizer updates model weights during training.
Sparse categorical cross-entropy is used as the loss function
because the target labels are integers from 0 to 9.


In [ ]:

model_nn = Sequential([
    Input(shape=(784,)),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

model_nn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model_nn.fit(
    X_train_flat,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    verbose=1
)



## 7. Evaluate the Neural Network

The trained neural network is evaluated using the test dataset.
The output layer produces ten class probabilities through Softmax.
The predicted digit is selected using the highest probability.

The classification report provides precision, recall, F1-score,
and support for each digit class.


In [ ]:

test_loss, test_acc = model_nn.evaluate(
    X_test_flat, y_test, verbose=0
)

probabilities = model_nn.predict(X_test_flat, verbose=0)
y_pred_nn = np.argmax(probabilities, axis=1)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn, digits=4))



## 8. Training and Validation Accuracy

Training accuracy measures the performance on training data.
Validation accuracy measures the performance on the validation
subset that is not used for weight updates.

A large difference between training and validation accuracy
may indicate overfitting. Increasing accuracy across both
curves generally indicates that the model is learning.


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()



## 9. Comparison of Both Models

Both models are evaluated on the same test dataset using
accuracy, macro precision, macro recall, and macro F1-score.

The comparison helps determine which model performs better
on handwritten digit recognition.


In [ ]:

def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1-score": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
    }

lr_metrics = calculate_metrics(y_test, y_pred_lr)
nn_metrics = calculate_metrics(y_test, y_pred_nn)

print(f"{'Metric':<20}{'Logistic Regression':>23}{'Neural Network':>20}")

for metric in lr_metrics:
    print(
        f"{metric:<20}"
        f"{lr_metrics[metric]:>23.4f}"
        f"{nn_metrics[metric]:>20.4f}"
    )

plt.figure(figsize=(7, 5))
plt.bar(
    ["Logistic Regression", "Dense Neural Network"],
    [lr_metrics["Accuracy"], nn_metrics["Accuracy"]]
)

plt.ylabel("Test Accuracy")
plt.title("Model Accuracy Comparison")
plt.ylim(0, 1)
plt.show()



### Result Interpretation

The model with higher test accuracy and stronger macro
precision, recall, and F1-score has better overall performance.

The Dense Neural Network is expected to perform better because
it can learn nonlinear patterns and complex combinations of
pixels. The final conclusion should be based on the actual
metrics printed during execution.

## 10. Result

The MNIST handwritten digit recognition experiment was
implemented using Logistic Regression and a Dense Neural
Network. Both models were trained and evaluated on the test
dataset. Their performance was compared using accuracy,
macro precision, macro recall, macro F1-score, and a
confusion matrix.

## 11. Conclusion

The experiment demonstrates the application of multiclass
classification to handwritten digit recognition. Data
normalization, flattening, model training, and evaluation
were performed. Comparing the baseline model with the neural
network illustrates how nonlinear learning can improve
classification performance.

## 12. Viva Questions and Answers

**Q1. Why is MNIST a multiclass classification problem?**

MNIST has ten possible target classes, representing digits 0 to 9.

**Q2. What is the difference between features and target?**

Features are the pixel values of an image. The target is the
actual digit label.

**Q3. Why are pixel values normalized?**

Normalization scales pixel values between 0 and 1, helping
neural networks train more effectively.

**Q4. Why is Softmax used in the output layer?**

Softmax converts output scores into probabilities for the
ten classes, summing to 1.

**Q5. What does a diagonal value in a confusion matrix represent?**

It represents the number of correctly classified samples
for the corresponding class.

**Q6. What is macro average?**

Macro average calculates a metric independently for every class
and takes the unweighted mean of the ten values.

**Q7. What is Dropout?**

Dropout randomly disables a fraction of neurons during training
to help reduce overfitting.

**Q8. Why is Adam used?**

Adam is an adaptive optimization algorithm that adjusts
parameter updates using estimates of gradients and their moments.

**Q9. What is the difference between accuracy and F1-score?**

Accuracy measures the proportion of correct predictions.
F1-score is the harmonic mean of precision and recall.

**Q10. Why can high accuracy hide weak class performance?**

A model may perform poorly on individual classes despite
high overall accuracy. Per-class metrics reveal these weaknesses.


